In [1]:
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
FILES = {
    'BTC': {
        'binance': './notebooks/download_notebook/binance_spot/btcusdt_spot_1h.parquet',
        'ccdata': './notebooks/download_notebook/ccdata/btc_ccdata_1h.parquet'
    },
    'ETH': {
        'binance': './notebooks/download_notebook/binance_spot/ethusdt_spot_1h.parquet',
        'ccdata': './notebooks/download_notebook/ccdata/eth_ccdata_1h.parquet'
    }
}

TIME_COLUMN = 'date' # Modifiez si la colonne s'appelle 'timestamp' ou autre
FREQ = '1H'          # Fréquence horaire

def load_and_prepare(filepath, source_name):
    """Charge le parquet et configure l'index temporel."""
    df = pd.read_parquet(filepath)
    # S'assurer que la colonne est bien en datetime UTC
    df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], utc=True)
    df = df.set_index(TIME_COLUMN).sort_index()
    
    # Ajout d'un préfixe pour différencier les colonnes lors de la fusion
    df = df.add_prefix(f'{source_name}_')
    return df

def analyze_asset_quality(asset, paths):
    print(f"\n{'='*50}")
    print(f"ANALYSE DE QUALITÉ DES DONNÉES : {asset}")
    print(f"{'='*50}")

    # 1. Chargement
    df_binance = load_and_prepare(paths['binance'], 'binance')
    df_ccdata = load_and_prepare(paths['ccdata'], 'ccdata')

    # 2. Création du MASTER CALENDAR (Référence stricte)
    # On prend la date de début la plus ancienne et la date de fin la plus récente des 2 datasets
    start_date = min(df_binance.index.min(), df_ccdata.index.min())
    end_date = max(df_binance.index.max(), df_ccdata.index.max())
    
    master_calendar = pd.date_range(start=start_date, end=end_date, freq=FREQ, tz='UTC')
    df_master = pd.DataFrame(index=master_calendar)
    df_master.index.name = TIME_COLUMN

    print(f"Période analysée : du {start_date} au {end_date}")
    print(f"Total des heures attendues (Master Calendar) : {len(df_master)}")

    # 3. Fusion (Left Join) sur le Master Calendar
    # Cela permet de révéler les trous horaires qui n'existent pas du tout dans les datasets d'origine
    df_merged = df_master.join(df_binance, how='left').join(df_ccdata, how='left')

    # 4. Analyse des valeurs manquantes (Trous)
    missing_binance = df_merged['binance_close'].isna().sum()
    missing_ccdata = df_merged['ccdata_close'].isna().sum()
    
    print("\n--- 1. EXHAUSTIVITÉ (Données manquantes) ---")
    print(f"Binance : {missing_binance} heures manquantes ({(missing_binance/len(df_master))*100:.2f}%)")
    print(f"CCData  : {missing_ccdata} heures manquantes ({(missing_ccdata/len(df_master))*100:.2f}%)")

    # 5. Calcul des Rendements et Différences (sur les données communes)
    # Calcul des rendements logarithmiques horaires (mesure standard en recherche)
    df_merged['binance_return'] = np.log(df_merged['binance_close'] / df_merged['binance_close'].shift(1))
    df_merged['ccdata_return'] = np.log(df_merged['ccdata_close'] / df_merged['ccdata_close'].shift(1))

    # Écart absolu entre les deux prix
    df_merged['price_diff_abs'] = (df_merged['binance_close'] - df_merged['ccdata_close']).abs()
    df_merged['price_diff_pct'] = df_merged['price_diff_abs'] / df_merged['ccdata_close'] * 100

    print("\n--- 2. STATISTIQUES DESCRIPTIVES (Prix de Clôture) ---")
    desc = df_merged[['binance_close', 'ccdata_close']].describe().T
    print(desc[['mean', 'std', 'min', 'max']])

    print("\n--- 3. ÉCARTS DE PRIX (Binance vs CCData) ---")
    print(f"Écart moyen absolu : {df_merged['price_diff_abs'].mean():.4f} $")
    print(f"Écart moyen en %   : {df_merged['price_diff_pct'].mean():.4f} %")
    print(f"Écart maximum en % : {df_merged['price_diff_pct'].max():.4f} %")

    print("\n--- 4. CORRÉLATION ---")
    # Corrélation de Pearson sur les prix
    corr_price = df_merged['binance_close'].corr(df_merged['ccdata_close'])
    # Corrélation sur les rendements horaires (beaucoup plus robuste pour évaluer le comportement)
    corr_return = df_merged['binance_return'].corr(df_merged['ccdata_return'])
    
    print(f"Corrélation des prix (Close) : {corr_price:.6f}")
    print(f"Corrélation des rendements horaires : {corr_return:.6f}")
    print("\n")

# --- EXÉCUTION ---
for asset, paths in FILES.items():
    analyze_asset_quality(asset, paths)


ANALYSE DE QUALITÉ DES DONNÉES : BTC


/var/folders/lb/jdd7xv1d5xlfx__q1rzq3fvr0000gn/T/ipykernel_1693/313307257.py:44: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  master_calendar = pd.date_range(start=start_date, end=end_date, freq=FREQ, tz='UTC')


Période analysée : du 2021-03-15 00:00:00+00:00 au 2025-12-01 00:00:00+00:00
Total des heures attendues (Master Calendar) : 41329

--- 1. EXHAUSTIVITÉ (Données manquantes) ---
Binance : 12 heures manquantes (0.03%)
CCData  : 1 heures manquantes (0.00%)

--- 2. STATISTIQUES DESCRIPTIVES (Prix de Clôture) ---
                       mean           std       min        max
binance_close  54278.503083  29823.995053  15649.52  126011.18
ccdata_close   54285.061663  29826.049428  15631.36  126098.78

--- 3. ÉCARTS DE PRIX (Binance vs CCData) ---
Écart moyen absolu : 25.0819 $
Écart moyen en %   : 0.0488 %
Écart maximum en % : 2.5791 %

--- 4. CORRÉLATION ---
Corrélation des prix (Close) : 0.999999
Corrélation des rendements horaires : 0.997292



ANALYSE DE QUALITÉ DES DONNÉES : ETH
Période analysée : du 2021-03-15 00:00:00+00:00 au 2025-12-01 00:00:00+00:00
Total des heures attendues (Master Calendar) : 41329

--- 1. EXHAUSTIVITÉ (Données manquantes) ---
Binance : 12 heures manquantes (0.03%

/var/folders/lb/jdd7xv1d5xlfx__q1rzq3fvr0000gn/T/ipykernel_1693/313307257.py:44: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  master_calendar = pd.date_range(start=start_date, end=end_date, freq=FREQ, tz='UTC')


In [5]:
TIME_COLUMN = 'date' 
MASTER_FILE = './notebooks/notebook_01/analysis/windows/master_calendar_1h.parquet'

# --- 2. FONCTIONS DE CHARGEMENT ---
def load_and_prepare(filepath, source_name):
    """Charge le parquet et configure l'index temporel."""
    df = pd.read_parquet(filepath)
    df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], utc=True)
    df = df.set_index(TIME_COLUMN).sort_index()
    # On isole uniquement le prix de clôture pour l'analyse
    df = df[['close']].rename(columns={'close': f'{source_name}_close'})
    return df

def get_master_calendar(filepath):
    """Charge le Master Calendar qui sert de référentiel absolu."""
    df_master = pd.read_parquet(filepath)
    df_master[TIME_COLUMN] = pd.to_datetime(df_master[TIME_COLUMN], utc=True)
    df_master = df_master.set_index(TIME_COLUMN).sort_index()
    # On s'assure qu'il n'y a pas de doublons dans le calendrier
    df_master = df_master[~df_master.index.duplicated(keep='first')]
    return df_master

# --- 3. ANALYSE ---
def analyze_with_master(asset, paths, df_master):
    print(f"\n{'='*55}")
    print(f"ANALYSE DE QUALITÉ SOUS MASTER CALENDAR : {asset}")
    print(f"{'='*55}")

    # Chargement des sources
    df_binance = load_and_prepare(paths['binance'], 'binance')
    df_ccdata = load_and_prepare(paths['ccdata'], 'ccdata')

    # FUSION SUR LE MASTER CALENDAR (Left Join)
    # df_master dicte les règles : si une date n'est pas dans le master, elle est ignorée.
    # Si une date est dans le master mais pas dans les sources, un NaN (trou) est créé.
    df_merged = df_master.copy()
    df_merged = df_merged.join(df_binance, how='left').join(df_ccdata, how='left')

    total_master_hours = len(df_master)
    print(f"Période couverte : du {df_merged.index.min()} au {df_merged.index.max()}")
    print(f"Total des heures imposées par le Master : {total_master_hours}")

    # 1. Analyse de l'exhaustivité (Trous)
    missing_bin = df_merged['binance_close'].isna().sum()
    missing_cc = df_merged['ccdata_close'].isna().sum()
    
    print("\n[1] EXHAUSTIVITÉ DES DONNÉES (Valeurs manquantes)")
    print(f" - Binance : {missing_bin} heures manquantes ({(missing_bin/total_master_hours)*100:.4f}%)")
    print(f" - CCData  : {missing_cc} heures manquantes ({(missing_cc/total_master_hours)*100:.4f}%)")

    # 2. Synchronisation et Rendements horaires
    df_merged['binance_return'] = np.log(df_merged['binance_close'] / df_merged['binance_close'].shift(1))
    df_merged['ccdata_return'] = np.log(df_merged['ccdata_close'] / df_merged['ccdata_close'].shift(1))

    # 3. Différences de Prix
    df_merged['price_diff_abs'] = (df_merged['binance_close'] - df_merged['ccdata_close']).abs()
    df_merged['price_diff_pct'] = df_merged['price_diff_abs'] / df_merged['ccdata_close'] * 100

    print("\n[2] ÉCARTS DE PRIX (Binance vs CCData)")
    print(f" - Écart moyen absolu : {df_merged['price_diff_abs'].mean():.4f} $")
    print(f" - Écart moyen en %   : {df_merged['price_diff_pct'].mean():.6f} %")
    print(f" - Écart maximum en % : {df_merged['price_diff_pct'].max():.4f} %")

    print("\n[3] CORRÉLATIONS (Synchronisation)")
    corr_price = df_merged['binance_close'].corr(df_merged['ccdata_close'])
    corr_return = df_merged['binance_return'].corr(df_merged['ccdata_return'])
    
    print(f" - Corrélation des prix       : {corr_price:.6f}")
    print(f" - Corrélation des rendements : {corr_return:.6f} <-- (L'indicateur le plus critique)")
    
    # 4. Détection des plus gros désaccords
    if not df_merged['price_diff_pct'].isna().all():
        max_diff_time = df_merged['price_diff_pct'].idxmax()
        print(f"\n[!] Note : Le plus grand écart de prix ({df_merged['price_diff_pct'].max():.2f}%) a eu lieu le {max_diff_time}")
    print("\n")

# --- 4. EXÉCUTION ---
# On charge le Master Calendar une seule fois
master_calendar = get_master_calendar(MASTER_FILE)

for asset, paths in FILES.items():
    analyze_with_master(asset, paths, master_calendar)


ANALYSE DE QUALITÉ SOUS MASTER CALENDAR : BTC
Période couverte : du 2021-03-15 00:00:00+00:00 au 2025-11-30 23:00:00+00:00
Total des heures imposées par le Master : 41328

[1] EXHAUSTIVITÉ DES DONNÉES (Valeurs manquantes)
 - Binance : 12 heures manquantes (0.0290%)
 - CCData  : 0 heures manquantes (0.0000%)

[2] ÉCARTS DE PRIX (Binance vs CCData)
 - Écart moyen absolu : 25.0819 $
 - Écart moyen en %   : 0.048761 %
 - Écart maximum en % : 2.5791 %

[3] CORRÉLATIONS (Synchronisation)
 - Corrélation des prix       : 0.999999
 - Corrélation des rendements : 0.997292 <-- (L'indicateur le plus critique)

[!] Note : Le plus grand écart de prix (2.58%) a eu lieu le 2022-05-12 06:00:00+00:00



ANALYSE DE QUALITÉ SOUS MASTER CALENDAR : ETH
Période couverte : du 2021-03-15 00:00:00+00:00 au 2025-11-30 23:00:00+00:00
Total des heures imposées par le Master : 41328

[1] EXHAUSTIVITÉ DES DONNÉES (Valeurs manquantes)
 - Binance : 12 heures manquantes (0.0290%)
 - CCData  : 0 heures manquantes (0.00